# 01. Ingeniería de Datos Avanzada — v3 (Investor Ready)

## 1. Introducción
Este cuaderno transforma datos crudos de partidos de La Liga en un **dataset estructurado de alto valor predictivo**.

### Mejoras v3
- Limpieza de duplicados en origen (temporada 2024)
- Validación de integridad del dataset antes de exportar
- Diccionario de variables actualizado


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas.')

Librerías cargadas.


## 2. Carga y Limpieza de Datos

In [2]:
# --- CARGA ---
possible_paths = [
    'df_final_app.csv',
    '../df_final_app.csv',
    'Premier/df_final_app.csv',
]
CSV_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        CSV_PATH = p
        break
if not CSV_PATH:
    raise FileNotFoundError('No se encontró df_final_app.csv')

df = pd.read_csv(CSV_PATH)
df['Date'] = pd.to_datetime(df['Date'])
print(f'Filas cargadas: {len(df)}')

# --- DEDUPLICACIÓN CRÍTICA ---
before = len(df)
df = df.drop_duplicates(subset=['Date', 'HomeTeam', 'AwayTeam'], keep='first').reset_index(drop=True)
removed = before - len(df)
if removed > 0:
    print(f'⚠️ Eliminados {removed} duplicados → {len(df)} filas limpias')
else:
    print(f'✅ Sin duplicados. Total: {len(df)} partidos')

print(f'Temporadas: {sorted(df["Season"].unique())}')
print(f'Partidos por temporada: {df.groupby("Season").size().to_dict()}')

Filas cargadas: 8360
⚠️ Eliminados 2660 duplicados → 5700 filas limpias
Temporadas: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Partidos por temporada: {2010: 380, 2011: 380, 2012: 380, 2013: 380, 2014: 380, 2015: 380, 2016: 380, 2017: 380, 2018: 380, 2019: 380, 2020: 380, 2021: 380, 2022: 380, 2023: 380, 2024: 380}


## 3. Validación de Integridad

In [3]:
# Verificar columnas clave
required_cols = ['Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTR',
                 'B365H', 'B365D', 'B365A',
                 'Home_Elo', 'Away_Elo',
                 'Home_xG_Avg_L5', 'Away_xG_Avg_L5',
                 'Home_Streak_L5', 'Away_Streak_L5']

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    print(f'❌ Columnas faltantes: {missing_cols}')
else:
    print('✅ Todas las columnas requeridas presentes')

# Nulos
null_summary = df[required_cols].isnull().sum()
null_summary = null_summary[null_summary > 0]
if len(null_summary) == 0:
    print('✅ Sin valores nulos en columnas clave')
else:
    print(f'⚠️ Nulos encontrados:\n{null_summary}')

# Cuotas válidas (>= 1.01)
for col in ['B365H', 'B365D', 'B365A']:
    bad = (df[col] < 1.01).sum()
    if bad > 0:
        print(f'⚠️ {bad} cuotas inválidas en {col}')

print(f'\n📊 Distribución de resultados:')
print(df['FTR'].value_counts())

✅ Todas las columnas requeridas presentes
✅ Sin valores nulos en columnas clave

📊 Distribución de resultados:
FTR
H    2558
A    1773
D    1369
Name: count, dtype: int64


## 4. Exportación del Dataset Limpio

In [4]:
# Mapear target
target_map = {'A': 0, 'D': 1, 'H': 2}
df['Target'] = df['FTR'].map(target_map)

# Ordenar por fecha
df = df.sort_values('Date').reset_index(drop=True)

# Exportar
out_path = 'df_final_clean.csv'
df.to_csv(out_path, index=False)
print(f'✅ Dataset limpio exportado: {out_path}')
print(f'   Filas: {len(df)} | Columnas: {len(df.columns)}')
print(f'   Temporadas: {df["Season"].min()} - {df["Season"].max()}')

✅ Dataset limpio exportado: df_final_clean.csv
   Filas: 5700 | Columnas: 25
   Temporadas: 2010 - 2024
